In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in current working directory)
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Convert timestamp to timezone-aware datetime in UTC+8 (Asia/Shanghai)
# Treat the epoch seconds as local to Asia/Shanghai per instructions (tz_localize)
for df in (metric, trace, log, error_logs):
    if 'timestamp' in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)

# ---------- metric.csv summary ----------
metric_count = metric.shape[0]
metric_cmdb_unique = pd.Series(metric['cmdb_id'].dropna().unique()).head(50).reset_index(drop=True).to_frame(name='cmdb_id')

# Ensure numeric 'value'
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Aggregate by (cmdb_id, kpi_name): global P95 and count
metric_grp = (
    metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(p95_value=lambda x: np.nanpercentile(x.dropna(), 95) if x.dropna().size>0 else np.nan,
         count='count')
    .reset_index()
)
metric_top20 = metric_grp.sort_values('p95_value', ascending=False).head(20).reset_index(drop=True)

# ---------- trace.csv summary ----------
trace_count = trace.shape[0]
trace_cmdb_unique = pd.Series(trace['cmdb_id'].dropna().unique()).head(50).reset_index(drop=True).to_frame(name='cmdb_id')

trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

trace_grp = (
    trace.groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(p95_value=lambda x: np.nanpercentile(x.dropna(), 95) if x.dropna().size>0 else np.nan,
         count='count')
    .reset_index()
)
trace_top20 = trace_grp.sort_values('p95_value', ascending=False).head(20).reset_index(drop=True)

# ---------- log.csv summary ----------
log_count = log.shape[0]
log_cmdb_unique = pd.Series(log['cmdb_id'].dropna().unique()).head(50).reset_index(drop=True).to_frame(name='cmdb_id')

log['value'] = pd.to_numeric(log['value'], errors='coerce')

log_grp = (
    log.groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(p95_value=lambda x: np.nanpercentile(x.dropna(), 95) if x.dropna().size>0 else np.nan,
         count='count')
    .reset_index()
)
log_top20 = log_grp.sort_values('p95_value', ascending=False).head(20).reset_index(drop=True)

# ---------- error_logs.csv summary ----------
error_count = error_logs.shape[0]
error_cmdb_unique = pd.Series(error_logs['cmdb_id'].dropna().unique()).head(50).reset_index(drop=True).to_frame(name='cmdb_id')

# earliest 20 rows by timestamp
if 'timestamp_dt' in error_logs.columns:
    error_earliest20 = error_logs.sort_values('timestamp').head(20).loc[:, ['timestamp_dt', 'cmdb_id', 'message']].reset_index(drop=True)
    # rename timestamp column to be compact and clear
    error_earliest20 = error_earliest20.rename(columns={'timestamp_dt': 'timestamp(UTC+8)'})
else:
    error_earliest20 = error_logs.sort_values('timestamp').head(20).loc[:, ['timestamp', 'cmdb_id', 'message']].reset_index(drop=True)

# Compact summaries (limited rows shown)
metric_count, metric_cmdb_unique, metric_top20[['cmdb_id','kpi_name','p95_value','count']], \
trace_count, trace_cmdb_unique, trace_top20[['cmdb_id','trace_name','p95_value','count']], \
log_count, log_cmdb_unique, log_top20[['cmdb_id','log_name','p95_value','count']], \
error_count, error_cmdb_unique, error_earliest20

```
Out[1]:
```
```python
# Summarize the previously computed execution results in plain English.
# This is a concise, human-readable summary using variables already present in the kernel.

def fmt_num(x):
    try:
        # compact numeric formatting
        return f"{float(x):.6g}"
    except Exception:
        return str(x)

lines = []
lines.append(f"metric.csv: {metric_count} rows; unique cmdb_id (top 20): {metric_cmdb_unique['cmdb_id'].head(20).tolist()}")
lines.append("Top metric groups by global P95 (top 5 of returned top20):")
for _, row in metric_top20[['cmdb_id','kpi_name','p95_value','count']].head(5).iterrows():
    lines.append(f" - {row['cmdb_id']} | {row['kpi_name']} | P95={fmt_num(row['p95_value'])} | count={int(row['count'])}")

lines.append("") 
lines.append(f"trace.csv: {trace_count} rows; unique cmdb_id (top 20): {trace_cmdb_unique['cmdb_id'].head(20).tolist()}")
lines.append("Top trace groups by global P95 (top 5 of returned top20):")
for _, row in trace_top20[['cmdb_id','trace_name','p95_value','count']].head(5).iterrows():
    lines.append(f" - {row['cmdb_id']} | {row['trace_name']} | P95={fmt_num(row['p95_value'])} | count={int(row['count'])}")

lines.append("")
lines.append(f"log.csv: {log_count} rows; unique cmdb_id (top 20): {log_cmdb_unique['cmdb_id'].head(20).tolist()}")
lines.append("Top log groups by global P95 (top 5 of returned top20):")
for _, row in log_top20[['cmdb_id','log_name','p95_value','count']].head(5).iterrows():
    lines.append(f" - {row['cmdb_id']} | {row['log_name']} | P95={fmt_num(row['p95_value'])} | count={int(row['count'])}")

lines.append("")
# error_logs summary
earliest_ts = error_earliest20.iloc[0]['timestamp(UTC+8)'] if not error_earliest20.empty else "N/A"
lines.append(f"error_logs.csv: {error_count} rows; unique cmdb_id (top 20): {error_cmdb_unique['cmdb_id'].head(20).tolist()}")
lines.append(f"Earliest error log timestamp (UTC+8) in sample: {earliest_ts}")
lines.append("Earliest 3 error log entries (timestamp, cmdb_id, first ~80 chars of message):")
for _, r in error_earliest20.head(3).iterrows():
    msg = str(r['message']).replace("\n"," ")
    lines.append(f" - {r['timestamp(UTC+8)']} | {r['cmdb_id']} | {msg[:80]}{'...' if len(msg)>80 else ''}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(29735,      cmdb_id
0       IG01
1       IG02
2       MG01
3       MG02
4    Mysql01
..       ...
21  Tomcat03
22  Tomcat04
23  apache01
24  apache02
25  dockerA2

[26 rows x 1 columns],      cmdb_id                                           kpi_name     p95_value  count
0    Mysql02               Mysql-MySQL_3306_Innodb data written  2.823478e+12     30
1    Mysql02  Mysql-MySQL_3306_Innodb buffer pool read requests  1.665584e+12     30
2    Mysql02             Mysql-MySQL_3306_Innodb os log written  7.414639e+11     30
3    Mysql01               Mysql-MySQL_3306_Innodb data written  6.364775e+11     30
4    Mysql02                         Mysql-MySQL_3306_Rows Read  5.444795e+11     30
5    Mysql02                 Mysql-MySQL_3306_Handler Read Prev  5.442882e+11     30
6    Mysql02                        Mysql-MySQL_3306_Bytes Sent  1.899842e+11     30
7    Mysql01                        Mysql-MySQL_3306_Bytes Sent  7.930180e+10     30
8    Mysql02                  Mysql-MySQL_3306_Innodb data read  3.317295e+10     30
9    Mysql01             Mysql-MySQL_3306_Innodb os log written  9.312330e+09     30
10   Redis01  redis-Redis_6379_Redis  (total_commands_proces...  7.027456e+09     20
11   Redis02  redis-Redis_6379_Redis  (total_commands_proces...  6.168248e+09     20
12   Redis01            redis-Redis_6379_Redis  (keyspace_hits)  5.384557e+09     20
13   Redis02            redis-Redis_6379_Redis  (keyspace_hits)  4.553672e+09     20
14   Redis01                redis-Redis_6379_Redis  (maxmemory)  4.294967e+09     20
15   Redis02                redis-Redis_6379_Redis  (maxmemory)  4.294967e+09     20
16  Tomcat03           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory  4.187619e+09      6
17  Tomcat04           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory  4.187619e+09      6
18  Tomcat04             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory  4.187619e+09      6
19  Tomcat03             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory  4.187619e+09      6, 7014,      cmdb_id
0       IG01
1       IG02
2       MG01
3       MG02
4   Tomcat01
5   Tomcat02
6   Tomcat03
7   Tomcat04
8   dockerA1
9   dockerA2
10  dockerB1
11  dockerB2,      cmdb_id                     trace_name  p95_value  count
0   dockerB1           trace.self.row_count    1265.00     31
1   Tomcat03           trace.self.row_count     595.50     31
2   Tomcat01           trace.self.row_count     581.35     30
3   Tomcat02           trace.self.row_count     577.50     31
4   Tomcat04           trace.self.row_count     567.00     31
5   dockerA2           trace.self.row_count     435.00     31
6   dockerA1           trace.self.row_count     431.00     31
7   dockerB2           trace.self.row_count     429.50     31
8       IG02           trace.self.row_count     323.00     31
9       IG01           trace.self.row_count     316.00     31
10      MG01           trace.self.row_count     265.00     31
11      MG02           trace.self.row_count     259.50     31
12      MG01  trace.from_dockerB1.row_count      87.00     31
13  dockerB1        trace.to_MG01.row_count      87.00     31
14  dockerB1        trace.to_MG02.row_count      82.00     31
15      MG02  trace.from_dockerB1.row_count      82.00     31
16  dockerB1      trace.from_MG01.row_count      74.50     31
17      MG01    trace.to_dockerB1.row_count      74.50     31
18  dockerB1      trace.from_MG02.row_count      67.00     31
19      MG02    trace.to_dockerB1.row_count      67.00     31, 383,     cmdb_id
0  Tomcat01
1  Tomcat02
2  Tomcat03
3  Tomcat04
4  apache01
5  apache02,      cmdb_id          log_name  p95_value  count
0   apache01     log.row_count    162.500     31
1   apache02     log.row_count    158.500     31
2   Tomcat02     log.row_count     87.000     31
3   Tomcat03     log.row_count     86.000     31
4   Tomcat01     log.row_count     83.550     30
5   Tomcat04     log.row_count     83.500     31
6   apache02   log.error_count      6.000     31
7   apache01   log.error_count      3.000     31
8   Tomcat04  log.gc_pause_max      0.087      4
9   Tomcat01  log.gc_pause_max      0.087      3
10  Tomcat03  log.gc_pause_max      0.068      3
11  Tomcat02  log.gc_pause_max      0.067      3
12  Tomcat03   log.error_count      0.000     31
13  Tomcat02   log.error_count      0.000     31
14  Tomcat01   log.error_count      0.000     30
15  Tomcat04   log.error_count      0.000     31, 41,     cmdb_id
0  apache01
1  apache02,             timestamp(UTC+8)   cmdb_id                                            message
0  2021-03-24 23:40:08+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...
1  2021-03-24 23:40:27+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...
2  2021-03-24 23:40:31+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest7.json H...
3  2021-03-24 23:40:34+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...
4  2021-03-24 23:40:47+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest3.json...
5  2021-03-24 23:40:50+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json...
6  2021-03-24 23:40:56+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest6.json H...
7  2021-03-24 23:40:59+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest5.json...
8  2021-03-24 23:41:07+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest10.jso...
9  2021-03-24 23:41:11+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest11.json ...
10 2021-03-24 23:41:13+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest9.json...
11 2021-03-24 23:41:44+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...
12 2021-03-24 23:41:48+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest11.json ...
13 2021-03-24 23:41:54+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...
14 2021-03-24 23:41:59+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest2.json H...
15 2021-03-24 23:42:02+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest3.json...
16 2021-03-24 23:42:20+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest7.json H...
17 2021-03-24 23:42:24+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest2.json H...
18 2021-03-24 23:42:26+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H...
19 2021-03-24 23:42:27+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest3.json...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure metric DataFrame and necessary columns exist in the kernel
# (metric was loaded previously). If timestamp_dt missing, create it.
if 'timestamp_dt' not in metric.columns:
    metric['timestamp_dt'] = pd.to_datetime(metric['timestamp'], unit='s').dt.tz_localize(tz)

# Ensure numeric values
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, kpi_name) from full metric series (global threshold)
metric_global_p95 = (
    metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(p95_value=lambda x: np.nanpercentile(x.dropna(), 95) if x.dropna().size > 0 else np.nan)
    .reset_index()
)

# 2) Define incident window in UTC+8 (inclusive)
start = pd.Timestamp('2021-03-25 07:30:00').tz_localize(tz)
end = pd.Timestamp('2021-03-25 08:00:00').tz_localize(tz)

# 3) Filter metric rows inside the window
metric_window = metric[(metric['timestamp_dt'] >= start) & (metric['timestamp_dt'] <= end)].copy()

# 4) Merge global p95 into the window rows (do not recompute thresholds after filtering)
metric_window = metric_window.merge(metric_global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 5) Determine anomaly boolean per row: value >= p95_value (handle NaN p95 as False)
metric_window['is_anomaly'] = False
mask_valid = metric_window['p95_value'].notna() & metric_window['value'].notna()
metric_window.loc[mask_valid, 'is_anomaly'] = metric_window.loc[mask_valid, 'value'] >= metric_window.loc[mask_valid, 'p95_value']

# 6) Aggregate per group within the window
agg = (
    metric_window.groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        p95_value=('p95_value', 'first'),  # same per group
        count_in_window=('value', 'count'),
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp_dt', lambda x: x.loc[metric_window.loc[x.index, 'is_anomaly']].min() if metric_window.loc[x.index, 'is_anomaly'].any() else pd.NaT)
    )
    .reset_index()
)

# 7) Keep only groups with anomaly_count > 0, sort and limit to top 20
anomalies = agg[agg['anomaly_count'] > 0].copy()
anomalies = anomalies.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Convert earliest_anomaly_timestamp to compact string (UTC+8)
anomalies['earliest_anomaly_timestamp(UTC+8)'] = anomalies['earliest_anomaly_timestamp'].astype(str)

# Select and order required columns
anomalies_top20 = anomalies[['cmdb_id', 'kpi_name', 'p95_value', 'count_in_window', 'anomaly_count', 'earliest_anomaly_timestamp(UTC+8)']].copy()

# 8) Scalars: total groups with anomalies and total anomaly points across groups (within the window)
total_groups_with_anomalies = int(agg['anomaly_count'].gt(0).sum())
total_anomaly_points = int(agg['anomaly_count'].sum())

# Return compact results
anomalies_top20, total_groups_with_anomalies, total_anomaly_points

```
Out[2]:
```
```python
# Summarize the anomaly detection result in plain English using existing variables.
summary_lines = [
    "Summary of metric anomalies within 2021-03-25 07:30:00 to 08:00:00 (UTC+8):",
    "",
    " - No metric groups had anomalies in the specified window.",
    f" - total_groups_with_anomalies = {total_groups_with_anomalies}",
    f" - total_anomaly_points = {total_anomaly_points}",
    "",
    "Explanation: Using the global P95 thresholds computed from the full metric.csv series,",
    "no (cmdb_id, kpi_name) group had any data points within the incident window that met or",
    "exceeded its global P95, so the anomalies_top20 table is empty."
]
summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, kpi_name, p95_value, count_in_window, anomaly_count, earliest_anomaly_timestamp(UTC+8)]
Index: [], 0, 0)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz

# reuse existing trace DataFrame and tz if present; otherwise load/create timestamp_dt
tz = pytz.timezone('Asia/Shanghai')

# ensure trace has timestamp_dt and numeric value
if 'timestamp_dt' not in trace.columns:
    trace['timestamp_dt'] = pd.to_datetime(trace['timestamp'], unit='s').dt.tz_localize(tz)
else:
    # ensure tz-localized
    if trace['timestamp_dt'].dt.tz is None:
        trace['timestamp_dt'] = trace['timestamp_dt'].dt.tz_localize(tz)

trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# 1) Compute global thresholds (P95, P90, P5) per (cmdb_id, trace_name) using the full trace series
trace_global_thresh = (
    trace.groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(
        p95_value=lambda x: np.nanpercentile(x.dropna(), 95) if x.dropna().size>0 else np.nan,
        p90_value=lambda x: np.nanpercentile(x.dropna(), 90) if x.dropna().size>0 else np.nan,
        p5_value =lambda x: np.nanpercentile(x.dropna(), 5)  if x.dropna().size>0 else np.nan
    )
    .reset_index()
)

# 2) Define incident window in UTC+8 (inclusive)
start = pd.Timestamp('2021-03-25 07:30:00').tz_localize(tz)
end   = pd.Timestamp('2021-03-25 08:00:00').tz_localize(tz)

# 3) Filter trace rows inside the window
trace_window = trace[(trace['timestamp_dt'] >= start) & (trace['timestamp_dt'] <= end)].copy()

# 4) Merge global thresholds into the window rows
trace_window = trace_window.merge(trace_global_thresh, on=['cmdb_id', 'trace_name'], how='left')

# 5) Compute anomaly booleans per row
trace_window['is_high'] = False
trace_window['is_low'] = False
mask_valid = trace_window['value'].notna()

mask_high = mask_valid & trace_window['p95_value'].notna()
trace_window.loc[mask_high, 'is_high'] = trace_window.loc[mask_high, 'value'] >= trace_window.loc[mask_high, 'p95_value']

mask_low = mask_valid & trace_window['p5_value'].notna()
trace_window.loc[mask_low, 'is_low'] = trace_window.loc[mask_low, 'value'] <= trace_window.loc[mask_low, 'p5_value']

trace_window['is_any'] = trace_window['is_high'] | trace_window['is_low']

# 6) Aggregate per group within the window
agg_trace = (
    trace_window.groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(
        p95_value=('p95_value', 'first'),
        p90_value=('p90_value', 'first'),
        p5_value =('p5_value', 'first'),
        count_in_window=('value', 'count'),
        anomaly_count_high=('is_high', 'sum'),
        anomaly_count_low =('is_low', 'sum'),
        earliest_anomaly_timestamp=('timestamp_dt', lambda x: x.loc[trace_window.loc[x.index, 'is_any']].min() if trace_window.loc[x.index, 'is_any'].any() else pd.NaT)
    )
    .reset_index()
)

# 7) Select groups with any anomalies and sort by total anomaly count descending, limit top 20
agg_trace['total_anomalies'] = agg_trace['anomaly_count_high'] + agg_trace['anomaly_count_low']
anomalies_trace = agg_trace[agg_trace['total_anomalies'] > 0].copy()
anomalies_trace = anomalies_trace.sort_values('total_anomalies', ascending=False).head(20).reset_index(drop=True)

# format earliest timestamp column
anomalies_trace['earliest_anomaly_timestamp(UTC+8)'] = anomalies_trace['earliest_anomaly_timestamp'].astype(str)

# final compact columns
anomalies_top20_trace = anomalies_trace[
    ['cmdb_id','trace_name','p95_value','p90_value','p5_value','count_in_window',
     'anomaly_count_high','anomaly_count_low','earliest_anomaly_timestamp(UTC+8)']
].copy()

# Scalars
total_trace_groups_with_anomalies = int(agg_trace['total_anomalies'].gt(0).sum())
total_trace_anomaly_points = int(agg_trace['total_anomalies'].sum())

# Return compact results
anomalies_top20_trace, total_trace_groups_with_anomalies, total_trace_anomaly_points

```
Out[3]:
```
# Summary of trace anomaly analysis (plain English), using existing computed scalars.
summary_lines = [
    "Trace anomaly analysis for window 2021-03-25 07:30:00 to 08:00:00 (UTC+8):",
    "",
    " - Global thresholds (P95, P90, P5) were computed per (cmdb_id, trace_name) from the full trace.csv series.",
    " - Trace rows were filtered to the specified incident window and compared to those global thresholds.",
    "",
    f" - total_trace_groups_with_anomalies = {total_trace_groups_with_anomalies}",
    f" - total_trace_anomaly_points = {total_trace_anomaly_points}",
    "",
    "Conclusion: No trace groups had high or low anomalies in the specified window (no rows met value >= P95 or value <= P5), so the returned anomalies table is empty."
]
summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, trace_name, p95_value, p90_value, p5_value, count_in_window, anomaly_count_high, anomaly_count_low, earliest_anomaly_timestamp(UTC+8)]
Index: [], 0, 0)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp_dt exists and is timezone-aware for log and error_logs
for df in (log, error_logs):
    if 'timestamp_dt' not in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)
    else:
        if df['timestamp_dt'].dt.tz is None:
            df['timestamp_dt'] = df['timestamp_dt'].dt.tz_localize(tz)

# Ensure numeric values in log
log['value'] = pd.to_numeric(log['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, log_name) from full log series
log_global_thresh = (
    log.groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(
        p95_value=lambda x: np.nanpercentile(x.dropna(), 95) if x.dropna().size>0 else np.nan,
        p5_value =lambda x: np.nanpercentile(x.dropna(), 5)  if x.dropna().size>0 else np.nan
    )
    .reset_index()
)

# 2) Define incident window in UTC+8
start = pd.Timestamp('2021-03-25 07:30:00').tz_localize(tz)
end   = pd.Timestamp('2021-03-25 08:00:00').tz_localize(tz)

# Filter log rows to the window
log_window = log[(log['timestamp_dt'] >= start) & (log['timestamp_dt'] <= end)].copy()

# Merge global thresholds into window rows (do not recompute thresholds after filtering)
log_window = log_window.merge(log_global_thresh, on=['cmdb_id', 'log_name'], how='left')

# Compute anomaly booleans
log_window['is_high'] = False
log_window['is_low'] = False
valid_mask = log_window['value'].notna()

mask_high = valid_mask & log_window['p95_value'].notna()
log_window.loc[mask_high, 'is_high'] = log_window.loc[mask_high, 'value'] >= log_window.loc[mask_high, 'p95_value']

mask_low = valid_mask & log_window['p5_value'].notna()
log_window.loc[mask_low, 'is_low'] = log_window.loc[mask_low, 'value'] <= log_window.loc[mask_low, 'p5_value']

log_window['is_any'] = log_window['is_high'] | log_window['is_low']

# Aggregate per (cmdb_id, log_name) within the window
agg_log = (
    log_window.groupby(['cmdb_id', 'log_name'], dropna=False)
    .agg(
        p95_value=('p95_value', 'first'),
        p5_value =('p5_value', 'first'),
        count_in_window=('value', 'count'),
        anomaly_count_high=('is_high', 'sum'),
        anomaly_count_low =('is_low', 'sum'),
        earliest_anomaly_timestamp=('timestamp_dt', lambda x: x.loc[log_window.loc[x.index, 'is_any']].min() if log_window.loc[x.index, 'is_any'].any() else pd.NaT)
    )
    .reset_index()
)

# Keep only groups with any anomalies, sort by total anomalies desc, limit top 20
agg_log['total_anomalies'] = agg_log['anomaly_count_high'] + agg_log['anomaly_count_low']
anomalies_log = agg_log[agg_log['total_anomalies'] > 0].copy()
anomalies_log = anomalies_log.sort_values('total_anomalies', ascending=False).head(20).reset_index(drop=True)

# Format earliest timestamp column
anomalies_log['earliest_anomaly_timestamp(UTC+8)'] = anomalies_log['earliest_anomaly_timestamp'].astype(str)

# Final compact columns for logs
anomalies_top20_log = anomalies_log[
    ['cmdb_id','log_name','p95_value','p5_value','count_in_window',
     'anomaly_count_high','anomaly_count_low','earliest_anomaly_timestamp(UTC+8)']
].copy()

# Scalars for logs
total_log_groups_with_anomalies = int(agg_log['total_anomalies'].gt(0).sum())
total_log_anomaly_points = int(agg_log['total_anomalies'].sum())

# 3) Scan error_logs.csv within the same window
error_window = error_logs[(error_logs['timestamp_dt'] >= start) & (error_logs['timestamp_dt'] <= end)].copy()
error_window_total_count = int(error_window.shape[0])

# counts grouped by cmdb_id (top 20)
error_window_count_by_cmdb = (
    error_window.groupby('cmdb_id', dropna=False).size().reset_index(name='count')
    .sort_values('count', ascending=False).head(20).reset_index(drop=True)
)

# earliest 20 error log rows within the window
error_window_earliest20 = (
    error_window.sort_values('timestamp_dt').head(20)
    .loc[:, ['timestamp_dt','cmdb_id','message']].reset_index(drop=True)
)
# rename timestamp column for clarity and convert to string for compact display
error_window_earliest20 = error_window_earliest20.rename(columns={'timestamp_dt':'timestamp(UTC+8)'})
error_window_earliest20['timestamp(UTC+8)'] = error_window_earliest20['timestamp(UTC+8)'].astype(str)

# Return compact results
anomalies_top20_log, total_log_groups_with_anomalies, total_log_anomaly_points, error_window_total_count, error_window_count_by_cmdb, error_window_earliest20

```
Out[4]:
```
```python
# Summarize log and error log analysis in plain English using existing kernel variables.
lines = [
    "Log and error_log analysis for window 2021-03-25 07:30:00 to 08:00:00 (UTC+8):",
    "",
    "1) Logs (log.csv):",
    " - Global P95 and P5 thresholds were computed per (cmdb_id, log_name) from the full log series.",
    " - Rows within the incident window were compared against those global thresholds.",
    f" - total_log_groups_with_anomalies = {total_log_groups_with_anomalies}",
    f" - total_log_anomaly_points = {total_log_anomaly_points}",
    " - Result: No log groups exceeded the global P95 or fell below the global P5 within the window (no anomalies found).",
    "",
    "2) Error logs (error_logs.csv):",
    f" - total error log rows in window = {error_window_total_count}",
    " - counts by cmdb_id in window: none (no error logs found in the window).",
    " - earliest error log rows in window: none (no entries).",
    "",
    "Conclusion: Within the specified 30-minute incident window there were no detected log metric anomalies and no error_log entries."
]
summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, log_name, p95_value, p5_value, count_in_window, anomaly_count_high, anomaly_count_low, earliest_anomaly_timestamp(UTC+8)]
Index: [], 0, 0, 0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp(UTC+8), cmdb_id, message]
Index: [])```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')
expected_points = 31  # per instructions

def summarize_file_window(df, name):
    # Ensure timestamp_dt exists and is tz-aware UTC+8
    if 'timestamp_dt' not in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)
    else:
        if df['timestamp_dt'].dt.tz is None:
            df['timestamp_dt'] = df['timestamp_dt'].dt.tz_localize(tz)
    # Use integer timestamp (seconds) for counting unique timestamps (they are floored to minute)
    if 'timestamp' not in df.columns:
        df['timestamp'] = df['timestamp_dt'].astype('int64') // 10**9

    # Define window boundaries in UTC+8
    start = pd.Timestamp('2021-03-25 07:30:00').tz_localize(tz)
    end   = pd.Timestamp('2021-03-25 08:00:00').tz_localize(tz)
    start_s = int(start.value // 10**9)
    end_s = int(end.value // 10**9)

    # Filter to window
    df_window = df[(df['timestamp'] >= start_s) & (df['timestamp'] <= end_s)].copy()

    # All cmdb_ids present in the full file
    all_cmdb = pd.Series(df['cmdb_id'].dropna().unique(), name='cmdb_id').to_frame()

    # Aggregation per cmdb_id in window
    agg = (
        df_window.groupby('cmdb_id', dropna=False)
        .agg(
            count_in_window=('timestamp', 'count'),
            unique_timestamps_in_window=('timestamp', 'nunique')
        )
        .reset_index()
    )

    # Merge to include cmdb_ids with zero rows (left join from all_cmdb)
    summary = all_cmdb.merge(agg, on='cmdb_id', how='left').fillna(0)
    summary['count_in_window'] = summary['count_in_window'].astype(int)
    summary['unique_timestamps_in_window'] = summary['unique_timestamps_in_window'].astype(int)
    summary['fraction_of_expected'] = summary['unique_timestamps_in_window'] / expected_points

    # Sort by fraction ascending and limit to top20
    summary_sorted = summary.sort_values('fraction_of_expected', ascending=True).reset_index(drop=True)
    summary_top20 = summary_sorted[['cmdb_id','count_in_window','unique_timestamps_in_window','fraction_of_expected']].head(20).copy()

    # Components with zero points in window
    zero_mask = summary['count_in_window'] == 0
    zero_list = summary.loc[zero_mask, ['cmdb_id']].reset_index(drop=True)
    zero_count = int(zero_list.shape[0])

    # Keep only top20 zero cmdb list for compactness
    zero_list_top20 = zero_list.head(20)

    return summary_top20, zero_count, zero_list_top20

# Reuse existing metric, trace, log DataFrames in kernel
metric_summary, metric_zero_count, metric_zero_list = summarize_file_window(metric, 'metric')
trace_summary, trace_zero_count, trace_zero_list = summarize_file_window(trace, 'trace')
log_summary, log_zero_count, log_zero_list = summarize_file_window(log, 'log')

# Return compact results (use variable names, separated by commas)
metric_summary, metric_zero_count, metric_zero_list, trace_summary, trace_zero_count, trace_zero_list, log_summary, log_zero_count, log_zero_list

```
Out[5]:
```
Summary (window = 2021-03-25 07:30:00 to 08:00:00 UTC+8; expected unique points = 31):

Metric.csv
- Top results (sorted by fraction_of_expected ascending, showing up to 20):
  - dockerA2: count_in_window=116, unique_timestamps_in_window=30, fraction_of_expected=0.9677
  - IG01, MG01, IG02, Mysql01, Mysql02, Redis01, MG02, ServiceTest1..ServiceTest11, Redis02, etc.: each had unique_timestamps_in_window=31, fraction_of_expected=1.0
- Components with zero points: 0 (no cmdb_id had zero rows in the window).

Trace.csv
- Top results (sorted by fraction_of_expected ascending, up to 20):
  - Tomcat01: count_in_window=453, unique_timestamps_in_window=30, fraction_of_expected=0.9677
  - IG01, MG01, IG02, MG02, Tomcat02..Tomcat04, dockerA1, dockerA2, dockerB1, dockerB2: each had unique_timestamps_in_window=31, fraction_of_expected=1.0
- Components with zero points: 0 (no cmdb_id had zero rows in the window).

Log.csv
- Top results (sorted by fraction_of_expected ascending, up to 20):
  - Tomcat01: count_in_window=63, unique_timestamps_in_window=30, fraction_of_expected=0.9677
  - Tomcat02..Tomcat04, apache01, apache02: each had unique_timestamps_in_window=31, fraction_of_expected=1.0
- Components with zero points: 0 (no cmdb_id had zero rows in the window).

Overall: Almost all components had the full set of expected timestamps (31) within the 30-minute window. A few components (dockerA2 in metrics, Tomcat01 in traces, Tomcat01 in logs) missed one timestamp (30/31 ≈ 0.9677). No components were completely missing data in this window for any of the three files.

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id  count_in_window  unique_timestamps_in_window  fraction_of_expected
0        dockerA2              116                           30              0.967742
1            IG01             1120                           31              1.000000
2            MG01             1624                           31              1.000000
3            IG02             1104                           31              1.000000
4         Mysql01             4784                           31              1.000000
5         Mysql02             4727                           31              1.000000
6         Redis01             1594                           31              1.000000
7            MG02             1614                           31              1.000000
8    ServiceTest1              124                           31              1.000000
9   ServiceTest10              124                           31              1.000000
10  ServiceTest11              124                           31              1.000000
11   ServiceTest2              124                           31              1.000000
12   ServiceTest3              124                           31              1.000000
13   ServiceTest4              124                           31              1.000000
14   ServiceTest5              124                           31              1.000000
15        Redis02             1829                           31              1.000000
16   ServiceTest6              124                           31              1.000000
17   ServiceTest7              124                           31              1.000000
18   ServiceTest9              124                           31              1.000000
19   ServiceTest8              124                           31              1.000000, 0, Empty DataFrame
Columns: [cmdb_id]
Index: [],      cmdb_id  count_in_window  unique_timestamps_in_window  fraction_of_expected
0   Tomcat01              453                           30              0.967742
1       IG01              462                           31              1.000000
2       MG01             1197                           31              1.000000
3       IG02              462                           31              1.000000
4       MG02             1200                           31              1.000000
5   Tomcat02              465                           31              1.000000
6   Tomcat03              465                           31              1.000000
7   Tomcat04              465                           31              1.000000
8   dockerA1              459                           31              1.000000
9   dockerA2              462                           31              1.000000
10  dockerB1              465                           31              1.000000
11  dockerB2              459                           31              1.000000, 0, Empty DataFrame
Columns: [cmdb_id]
Index: [],     cmdb_id  count_in_window  unique_timestamps_in_window  fraction_of_expected
0  Tomcat01               63                           30              0.967742
1  Tomcat02               65                           31              1.000000
2  Tomcat03               65                           31              1.000000
3  Tomcat04               66                           31              1.000000
4  apache01               62                           31              1.000000
5  apache02               62                           31              1.000000, 0, Empty DataFrame
Columns: [cmdb_id]
Index: [])```
```

In [ ]:
# In[6]:

import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure metric DataFrame has timezone-aware timestamp_dt and numeric value
if 'timestamp_dt' not in metric.columns:
    metric['timestamp_dt'] = pd.to_datetime(metric['timestamp'], unit='s').dt.tz_localize(tz)
else:
    if metric['timestamp_dt'].dt.tz is None:
        metric['timestamp_dt'] = metric['timestamp_dt'].dt.tz_localize(tz)

metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# 1) Compute global P90 and P10 per (cmdb_id, kpi_name) from the full metric series
metric_thresh = (
    metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(
        p90_value=lambda x: np.nanpercentile(x.dropna(), 90) if x.dropna().size>0 else np.nan,
        p10_value=lambda x: np.nanpercentile(x.dropna(), 10) if x.dropna().size>0 else np.nan
    )
    .reset_index()
)

# 2) Define incident window in UTC+8 (inclusive)
start = pd.Timestamp('2021-03-25 07:30:00').tz_localize(tz)
end   = pd.Timestamp('2021-03-25 08:00:00').tz_localize(tz)

# 3) Filter metric rows to the window
metric_window = metric[(metric['timestamp_dt'] >= start) & (metric['timestamp_dt'] <= end)].copy()

# 4) Merge global thresholds into the window rows (thresholds computed from full series)
metric_window = metric_window.merge(metric_thresh, on=['cmdb_id', 'kpi_name'], how='left')

# 5) Compute anomaly booleans using global thresholds (do not recompute thresholds)
metric_window['is_high'] = False
metric_window['is_low'] = False
valid_mask = metric_window['value'].notna()

mask_high = valid_mask & metric_window['p90_value'].notna()
metric_window.loc[mask_high, 'is_high'] = metric_window.loc[mask_high, 'value'] >= metric_window.loc[mask_high, 'p90_value']

mask_low = valid_mask & metric_window['p10_value'].notna()
metric_window.loc[mask_low, 'is_low'] = metric_window.loc[mask_low, 'value'] <= metric_window.loc[mask_low, 'p10_value']

metric_window['is_any'] = metric_window['is_high'] | metric_window['is_low']

# 6) Aggregate per group within the window
agg_counts = (
    metric_window.groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        p90_value=('p90_value', 'first'),
        p10_value=('p10_value', 'first'),
        count_in_window=('value', 'count'),
        anomaly_count_high=('is_high', 'sum'),
        anomaly_count_low=('is_low', 'sum')
    )
    .reset_index()
)

# earliest anomaly timestamp per group (only among rows flagged is_any)
earliest_ts = (
    metric_window[metric_window['is_any']]
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['timestamp_dt']
    .min()
    .reset_index()
    .rename(columns={'timestamp_dt': 'earliest_anomaly_timestamp'})
)

# Merge earliest timestamps into agg_counts
agg_counts = agg_counts.merge(earliest_ts, on=['cmdb_id', 'kpi_name'], how='left')

# 7) Keep only groups with any anomalies (high or low), sort by total anomalies desc, limit top 20
agg_counts['total_anomalies'] = agg_counts['anomaly_count_high'] + agg_counts['anomaly_count_low']
anomalies_metric = agg_counts[agg_counts['total_anomalies'] > 0].copy()
anomalies_metric = anomalies_metric.sort_values('total_anomalies', ascending=False).head(20).reset_index(drop=True)

# Format earliest timestamp for compact display
anomalies_metric['earliest_anomaly_timestamp(UTC+8)'] = anomalies_metric['earliest_anomaly_timestamp'].astype(str)

# Final compact columns
anomalies_top20_metric = anomalies_metric[
    ['cmdb_id','kpi_name','p90_value','p10_value','count_in_window',
     'anomaly_count_high','anomaly_count_low','earliest_anomaly_timestamp(UTC+8)']
].copy()

# Scalars
total_metric_groups_with_loose_anomalies = int(agg_counts['total_anomalies'].gt(0).sum())
total_metric_anomaly_points = int(agg_counts['total_anomalies'].sum())

# Return compact results
anomalies_top20_metric, total_metric_groups_with_loose_anomalies, total_metric_anomaly_points

```
Out[6]:
```
# Summarize the metric P90/P10 anomaly analysis in plain English using existing kernel variables.
summary_lines = [
    "Metric loose-anomaly analysis (using global P90/P10) for window 2021-03-25 07:30:00 to 08:00:00 (UTC+8):",
    "",
    " - Global P90 and P10 were computed per (cmdb_id, kpi_name) from the full metric series.",
    " - Metric rows in the specified window were checked against those global thresholds (value >= P90 or value <= P10).",
    "",
    " - Result: No metric groups had values in the window that met the loose-anomaly criteria.",
    f" - total_metric_groups_with_loose_anomalies = {total_metric_groups_with_loose_anomalies}",
    f" - total_metric_anomaly_points = {total_metric_anomaly_points}",
    "",
    "Conclusion: Using global P90/P10 thresholds, there were no high or low metric anomalies detected in the incident window; the anomalies table is empty."
]
summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, kpi_name, p90_value, p10_value, count_in_window, anomaly_count_high, anomaly_count_low, earliest_anomaly_timestamp(UTC+8)]
Index: [], 0, 0)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure metric DataFrame has timezone-aware timestamp_dt and numeric 'value'
if 'timestamp_dt' not in metric.columns:
    metric['timestamp_dt'] = pd.to_datetime(metric['timestamp'], unit='s').dt.tz_localize(tz)
else:
    if metric['timestamp_dt'].dt.tz is None:
        metric['timestamp_dt'] = metric['timestamp_dt'].dt.tz_localize(tz)

metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# 1) Compute historical mean and std per (cmdb_id, kpi_name) from the full series
group_stats = (
    metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(mean_value=lambda x: x.dropna().mean() if x.dropna().size>0 else np.nan,
         std_value =lambda x: x.dropna().std(ddof=0) if x.dropna().size>0 else np.nan)  # population std
    .reset_index()
)

# 2) Define incident window in UTC+8 (inclusive) and filter rows
start = pd.Timestamp('2021-03-25 07:30:00').tz_localize(tz)
end   = pd.Timestamp('2021-03-25 08:00:00').tz_localize(tz)
metric_window = metric[(metric['timestamp_dt'] >= start) & (metric['timestamp_dt'] <= end)].copy()

# 3) Merge stats into window rows (use historical stats computed from full series)
metric_window = metric_window.merge(group_stats, on=['cmdb_id', 'kpi_name'], how='left')

# Compute z = (value - mean)/std for rows where std_value > 0; otherwise z = NaN
metric_window['z'] = np.nan
valid_std_mask = metric_window['std_value'].notna() & (metric_window['std_value'] > 0) & metric_window['value'].notna()
metric_window.loc[valid_std_mask, 'z'] = (metric_window.loc[valid_std_mask, 'value'] - metric_window.loc[valid_std_mask, 'mean_value']) / metric_window.loc[valid_std_mask, 'std_value']

# Flag z anomalies
metric_window['z_pos'] = metric_window['z'] >= 3
metric_window['z_neg'] = metric_window['z'] <= -3
metric_window['z_any'] = metric_window['z_pos'] | metric_window['z_neg']

# 4) Aggregate per (cmdb_id, kpi_name)
agg_z = (
    metric_window.groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        count_in_window=('value', 'count'),
        z_anomaly_count_pos=('z_pos', 'sum'),
        z_anomaly_count_neg=('z_neg', 'sum'),
        max_abs_z_in_window=('z', lambda x: np.nanmax(np.abs(x)) if x.notna().any() else np.nan)
    )
    .reset_index()
)

# earliest z anomaly timestamp per group (only rows with z_any True)
earliest_z = (
    metric_window[metric_window['z_any']]
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['timestamp_dt']
    .min()
    .reset_index()
    .rename(columns={'timestamp_dt': 'earliest_z_anomaly_timestamp'})
)

agg_z = agg_z.merge(earliest_z, on=['cmdb_id', 'kpi_name'], how='left')

# Keep only groups with any z anomalies (pos or neg)
agg_z['total_z_anomalies'] = agg_z['z_anomaly_count_pos'] + agg_z['z_anomaly_count_neg']
anomalies_z = agg_z[agg_z['total_z_anomalies'] > 0].copy()

# Sort by total anomalies desc, then by max_abs_z_in_window desc, limit top 20
anomalies_z = anomalies_z.sort_values(['total_z_anomalies', 'max_abs_z_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

# Format earliest timestamp and select final columns
anomalies_z['earliest_z_anomaly_timestamp(UTC+8)'] = anomalies_z['earliest_z_anomaly_timestamp'].astype(str)
anomalies_z_top20 = anomalies_z[
    ['cmdb_id','kpi_name','count_in_window','z_anomaly_count_pos','z_anomaly_count_neg','max_abs_z_in_window','earliest_z_anomaly_timestamp(UTC+8)']
].copy()

# Scalars
total_metric_groups_with_z_anomalies = int(agg_z['total_z_anomalies'].gt(0).sum())
total_metric_z_anomaly_points = int(agg_z['total_z_anomalies'].sum())

# Return compact results (DataFrame limited to top20, plus two scalars)
anomalies_z_top20, total_metric_groups_with_z_anomalies, total_metric_z_anomaly_points

```
Out[7]:
```
```python
# Plain-English summary of the z-score anomaly analysis (using existing kernel variables)
lines = [
    "Metric z-score anomaly analysis for 2021-03-25 07:30:00 to 08:00:00 (UTC+8):",
    "",
    " - Historical mean and std were computed per (cmdb_id, kpi_name) from the full metric.csv series.",
    " - For rows in the incident window, z = (value - mean) / std was computed (groups with std == 0 were skipped).",
    "",
    f" - total_metric_groups_with_z_anomalies = {total_metric_groups_with_z_anomalies}",
    f" - total_metric_z_anomaly_points = {total_metric_z_anomaly_points}",
    "",
    "Conclusion: No groups in the specified window exhibited z-score anomalies (no rows with z >= 3 or z <= -3)."
]
summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, kpi_name, count_in_window, z_anomaly_count_pos, z_anomaly_count_neg, max_abs_z_in_window, earliest_z_anomaly_timestamp(UTC+8)]
Index: [], 0, 0)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure trace and log have timezone-aware timestamp_dt and numeric 'value'
for df, name, valcol in ((trace, 'trace', 'value'), (log, 'log', 'value')):
    if 'timestamp_dt' not in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)
    else:
        if df['timestamp_dt'].dt.tz is None:
            df['timestamp_dt'] = df['timestamp_dt'].dt.tz_localize(tz)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Incident window
start = pd.Timestamp('2021-03-25 07:30:00').tz_localize(tz)
end   = pd.Timestamp('2021-03-25 08:00:00').tz_localize(tz)

# --- TRACE ---
# 1) historical mean/std per (cmdb_id, trace_name) using full trace series
trace_stats = (
    trace.groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(
        mean_value=lambda x: x.dropna().mean() if x.dropna().size>0 else np.nan,
        std_value =lambda x: x.dropna().std(ddof=0) if x.dropna().size>0 else np.nan
    )
    .reset_index()
)

# 2) filter trace rows to window
trace_window = trace[(trace['timestamp_dt'] >= start) & (trace['timestamp_dt'] <= end)].copy()

# 3) merge stats and compute z where std>0
trace_window = trace_window.merge(trace_stats, on=['cmdb_id', 'trace_name'], how='left')
trace_window['z'] = np.nan
mask_valid = trace_window['value'].notna() & trace_window['std_value'].notna() & (trace_window['std_value'] > 0)
trace_window.loc[mask_valid, 'z'] = (trace_window.loc[mask_valid, 'value'] - trace_window.loc[mask_valid, 'mean_value']) / trace_window.loc[mask_valid, 'std_value']

trace_window['z_pos'] = trace_window['z'] >= 3
trace_window['z_neg'] = trace_window['z'] <= -3
trace_window['z_any'] = trace_window['z_pos'] | trace_window['z_neg']

# 4) aggregate per group
agg_trace_z = (
    trace_window.groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(
        count_in_window=('value', 'count'),
        z_anomaly_count_pos=('z_pos', 'sum'),
        z_anomaly_count_neg=('z_neg', 'sum'),
        max_abs_z_in_window=('z', lambda x: np.nanmax(np.abs(x)) if x.notna().any() else np.nan)
    )
    .reset_index()
)

# earliest z anomaly timestamp per group
earliest_trace_z = (
    trace_window[trace_window['z_any']]
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['timestamp_dt']
    .min()
    .reset_index()
    .rename(columns={'timestamp_dt': 'earliest_z_anomaly_timestamp'})
)

agg_trace_z = agg_trace_z.merge(earliest_trace_z, on=['cmdb_id', 'trace_name'], how='left')

# filter groups with any z anomalies and sort/limit
agg_trace_z['total_z_anomalies'] = agg_trace_z['z_anomaly_count_pos'] + agg_trace_z['z_anomaly_count_neg']
anomalies_trace_z = agg_trace_z[agg_trace_z['total_z_anomalies'] > 0].copy()
anomalies_trace_z = anomalies_trace_z.sort_values(['total_z_anomalies', 'max_abs_z_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)
anomalies_trace_z['earliest_z_anomaly_timestamp(UTC+8)'] = anomalies_trace_z['earliest_z_anomaly_timestamp'].astype(str)

anomalies_trace_z_top20 = anomalies_trace_z[
    ['cmdb_id','trace_name','count_in_window','z_anomaly_count_pos','z_anomaly_count_neg','max_abs_z_in_window','earliest_z_anomaly_timestamp(UTC+8)']
].copy()

total_trace_groups_with_z_anomalies = int(agg_trace_z['total_z_anomalies'].gt(0).sum())
total_trace_z_anomaly_points = int(agg_trace_z['total_z_anomalies'].sum())

# --- LOG ---
# 1) historical mean/std per (cmdb_id, log_name) using full log series
log_stats = (
    log.groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(
        mean_value=lambda x: x.dropna().mean() if x.dropna().size>0 else np.nan,
        std_value =lambda x: x.dropna().std(ddof=0) if x.dropna().size>0 else np.nan
    )
    .reset_index()
)

# 2) filter log rows to window
log_window = log[(log['timestamp_dt'] >= start) & (log['timestamp_dt'] <= end)].copy()

# 3) merge stats and compute z where std>0
log_window = log_window.merge(log_stats, on=['cmdb_id', 'log_name'], how='left')
log_window['z'] = np.nan
mask_valid = log_window['value'].notna() & log_window['std_value'].notna() & (log_window['std_value'] > 0)
log_window.loc[mask_valid, 'z'] = (log_window.loc[mask_valid, 'value'] - log_window.loc[mask_valid, 'mean_value']) / log_window.loc[mask_valid, 'std_value']

log_window['z_pos'] = log_window['z'] >= 3
log_window['z_neg'] = log_window['z'] <= -3
log_window['z_any'] = log_window['z_pos'] | log_window['z_neg']

# 4) aggregate per group
agg_log_z = (
    log_window.groupby(['cmdb_id', 'log_name'], dropna=False)
    .agg(
        count_in_window=('value', 'count'),
        z_anomaly_count_pos=('z_pos', 'sum'),
        z_anomaly_count_neg=('z_neg', 'sum'),
        max_abs_z_in_window=('z', lambda x: np.nanmax(np.abs(x)) if x.notna().any() else np.nan)
    )
    .reset_index()
)

# earliest z anomaly timestamp per group
earliest_log_z = (
    log_window[log_window['z_any']]
    .groupby(['cmdb_id', 'log_name'], dropna=False)['timestamp_dt']
    .min()
    .reset_index()
    .rename(columns={'timestamp_dt': 'earliest_z_anomaly_timestamp'})
)

agg_log_z = agg_log_z.merge(earliest_log_z, on=['cmdb_id', 'log_name'], how='left')

# filter groups with any z anomalies and sort/limit
agg_log_z['total_z_anomalies'] = agg_log_z['z_anomaly_count_pos'] + agg_log_z['z_anomaly_count_neg']
anomalies_log_z = agg_log_z[agg_log_z['total_z_anomalies'] > 0].copy()
anomalies_log_z = anomalies_log_z.sort_values(['total_z_anomalies', 'max_abs_z_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)
anomalies_log_z['earliest_z_anomaly_timestamp(UTC+8)'] = anomalies_log_z['earliest_z_anomaly_timestamp'].astype(str)

anomalies_log_z_top20 = anomalies_log_z[
    ['cmdb_id','log_name','count_in_window','z_anomaly_count_pos','z_anomaly_count_neg','max_abs_z_in_window','earliest_z_anomaly_timestamp(UTC+8)']
].copy()

total_log_groups_with_z_anomalies = int(agg_log_z['total_z_anomalies'].gt(0).sum())
total_log_z_anomaly_points = int(agg_log_z['total_z_anomalies'].sum())

# Return compact results
anomalies_trace_z_top20, total_trace_groups_with_z_anomalies, total_trace_z_anomaly_points, anomalies_log_z_top20, total_log_groups_with_z_anomalies, total_log_z_anomaly_points

```
Out[8]:
```
```python
# Plain-English summary of z-score anomaly checks for trace.csv and log.csv (using existing kernel variables).
lines = [
    "Z-score anomaly summary for window 2021-03-25 07:30:00 to 08:00:00 (UTC+8):",
    "",
    "Trace data:",
    f" - total_trace_groups_with_z_anomalies = {total_trace_groups_with_z_anomalies}",
    f" - total_trace_z_anomaly_points = {total_trace_z_anomaly_points}",
    " - Conclusion: No trace groups exhibited z-score anomalies (no rows with z >= 3 or z <= -3).",
    "",
    "Log data:",
    f" - total_log_groups_with_z_anomalies = {total_log_groups_with_z_anomalies}",
    f" - total_log_z_anomaly_points = {total_log_z_anomaly_points}",
    " - Conclusion: No log groups exhibited z-score anomalies (no rows with z >= 3 or z <= -3).",
    "",
    "Overall: Using historical mean/std and z-score threshold of 3, neither trace nor log files showed z anomalies in the incident window; both result tables are empty."
]
summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, trace_name, count_in_window, z_anomaly_count_pos, z_anomaly_count_neg, max_abs_z_in_window, earliest_z_anomaly_timestamp(UTC+8)]
Index: [], 0, 0, Empty DataFrame
Columns: [cmdb_id, log_name, count_in_window, z_anomaly_count_pos, z_anomaly_count_neg, max_abs_z_in_window, earliest_z_anomaly_timestamp(UTC+8)]
Index: [], 0, 0)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure metric DataFrame and timestamp_dt exist and are timezone-aware
if 'metric' not in globals():
    metric = pd.read_csv('metric.csv')
if 'timestamp_dt' not in metric.columns:
    metric['timestamp_dt'] = pd.to_datetime(metric['timestamp'], unit='s').dt.tz_localize(tz)
else:
    if metric['timestamp_dt'].dt.tz is None:
        metric['timestamp_dt'] = metric['timestamp_dt'].dt.tz_localize(tz)

metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Define windows (UTC+8)
baseline_start = pd.Timestamp('2021-03-25 06:30:00').tz_localize(tz)
baseline_end   = pd.Timestamp('2021-03-25 07:30:00').tz_localize(tz)
incident_start = pd.Timestamp('2021-03-25 07:30:00').tz_localize(tz)
incident_end   = pd.Timestamp('2021-03-25 08:00:00').tz_localize(tz)

# Filter rows for each window
baseline_df = metric[(metric['timestamp_dt'] >= baseline_start) & (metric['timestamp_dt'] <= baseline_end)].copy()
incident_df = metric[(metric['timestamp_dt'] >= incident_start) & (metric['timestamp_dt'] <= incident_end)].copy()

# Aggregate baseline: mean_baseline per group
baseline_stats = (
    baseline_df.groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .agg(mean_baseline=lambda x: x.dropna().mean() if x.dropna().size>0 else np.nan)
    .reset_index()
)

# Aggregate incident: mean_incident, count_in_incident, earliest_incident_timestamp
incident_stats = (
    incident_df.groupby(['cmdb_id','kpi_name'], dropna=False)
    .agg(
        mean_incident = ('value', lambda x: x.dropna().mean() if x.dropna().size>0 else np.nan),
        count_in_incident = ('value', 'count'),
        earliest_incident_timestamp = ('timestamp_dt', lambda x: x.min() if x.notna().any() else pd.NaT)
    )
    .reset_index()
)

# Merge baseline and incident stats
merged = baseline_stats.merge(incident_stats, on=['cmdb_id','kpi_name'], how='inner')

# Skip groups where mean_baseline == 0 or NaN, or count_in_incident < 5
merged = merged[merged['mean_baseline'].notna() & (merged['mean_baseline'] != 0)]
merged = merged[merged['count_in_incident'].fillna(0).astype(int) >= 5].copy()

# Compute relative change percent
merged['relative_change_pct'] = 100.0 * (merged['mean_incident'] - merged['mean_baseline']) / merged['mean_baseline']

# Filter groups with abs(relative_change_pct) >= 50%
large_changes = merged[merged['relative_change_pct'].abs() >= 50.0].copy()

# Format earliest incident timestamp for display
large_changes['earliest_incident_timestamp(UTC+8)'] = large_changes['earliest_incident_timestamp'].astype(str)

# Select required columns and sort by abs(relative_change_pct) descending, limit to top 20
large_changes['abs_rel_change'] = large_changes['relative_change_pct'].abs()
large_changes = large_changes.sort_values(['abs_rel_change'], ascending=[False]).head(20)

result_cols = [
    'cmdb_id','kpi_name','mean_baseline','mean_incident','count_in_incident',
    'relative_change_pct','earliest_incident_timestamp(UTC+8)'
]
anomalies_relative_top20 = large_changes[result_cols].reset_index(drop=True)

# Scalar: total groups with large relative change
total_groups_with_large_relative_change = int(merged['relative_change_pct'].abs().ge(50.0).sum())

# Display compact results
anomalies_relative_top20, total_groups_with_large_relative_change

```
Out[9]:
```
```python
# Plain-English summary of relative-change analysis (using existing kernel variables).
summary_lines = [
    "Relative change analysis (baseline 2021-03-25 06:30:00–07:30:00 UTC+8 vs incident 2021-03-25 07:30:00–08:00:00 UTC+8):",
    "",
    "- For each (cmdb_id, kpi_name) we computed mean_baseline and mean_incident and required at least 5 points in the incident window.",
    "- Groups with mean_baseline == 0 or with fewer than 5 incident points were skipped.",
    "",
    f"- total groups with |relative_change| >= 50% = {total_groups_with_large_relative_change}",
    "",
    "Conclusion: No (cmdb_id, kpi_name) groups met the criteria (abs(relative_change) >= 50% with count_in_incident >= 5). The returned table is empty."
]
summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, kpi_name, mean_baseline, mean_incident, count_in_incident, relative_change_pct, earliest_incident_timestamp(UTC+8)]
Index: [], 0)```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp_dt exists and is timezone-aware for trace and log
for df in (trace, log):
    if 'timestamp_dt' not in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)
    else:
        if df['timestamp_dt'].dt.tz is None:
            df['timestamp_dt'] = df['timestamp_dt'].dt.tz_localize(tz)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Incident window boundaries (UTC+8)
start = pd.Timestamp('2021-03-25 07:30:00').tz_localize(tz)
end   = pd.Timestamp('2021-03-25 08:00:00').tz_localize(tz)

# Helper to aggregate row_count series, compute stats, and window z-scores
def analyze_row_count(series_df, name_col, series_label):
    # select row_count rows
    sel = series_df[series_df[name_col].str.contains('row_count', na=False)].copy()
    # aggregate per-minute sum across all cmdb_id
    agg_full = sel.groupby('timestamp_dt', dropna=False)['value'].sum().reset_index().rename(columns={'value':'sum_value'})
    agg_full = agg_full.sort_values('timestamp_dt').reset_index(drop=True)

    # global mean and std (population std ddof=0) from full series
    mean_full = float(agg_full['sum_value'].mean(skipna=True)) if not agg_full['sum_value'].dropna().empty else np.nan
    std_full  = float(agg_full['sum_value'].std(ddof=0, skipna=True)) if not agg_full['sum_value'].dropna().empty else np.nan

    # filter to incident window
    agg_window = agg_full[(agg_full['timestamp_dt'] >= start) & (agg_full['timestamp_dt'] <= end)].copy()
    # compute z, handle std==0 or nan
    if np.isnan(std_full) or std_full == 0:
        agg_window['z'] = np.nan
    else:
        agg_window['z'] = (agg_window['sum_value'] - mean_full) / std_full

    # prepare display table limited to 31 rows
    display_df = agg_window[['timestamp_dt','sum_value','z']].copy()
    display_df = display_df.sort_values('timestamp_dt').head(31).reset_index(drop=True)
    display_df['timestamp(UTC+8)'] = display_df['timestamp_dt'].astype(str)
    display_df = display_df[['timestamp(UTC+8)','sum_value','z']]

    # earliest timestamps and counts for significant drop/spike
    drop_mask = display_df['z'] <= -2
    spike_mask = display_df['z'] >= 2

    drop_count = int(drop_mask.sum())
    spike_count = int(spike_mask.sum())

    earliest_drop = display_df.loc[drop_mask, 'timestamp(UTC+8)'].min() if drop_count>0 else None
    earliest_spike = display_df.loc[spike_mask, 'timestamp(UTC+8)'].min() if spike_count>0 else None

    # compact summary row
    summary = pd.DataFrame([{
        'series': series_label,
        'mean_full': mean_full,
        'std_full': std_full,
        'window_minutes': int(display_df.shape[0]),
        'earliest_drop_timestamp(UTC+8)': earliest_drop,
        'earliest_spike_timestamp(UTC+8)': earliest_spike,
        'drop_count_in_window': drop_count,
        'spike_count_in_window': spike_count
    }])

    return display_df, summary, agg_full

# Trace row_count analysis
trace_window_display, trace_summary, trace_agg_full = analyze_row_count(trace, 'trace_name', 'trace-row_count-sum')

# Log row_count analysis
log_window_display, log_summary, log_agg_full = analyze_row_count(log, 'log_name', 'log-row_count-sum')

# Return compact outputs: per-minute window tables and per-series summaries
trace_window_display, trace_summary, log_window_display, log_summary

```
Out[10]:
```
```python
# Summarize the trace and log row_count aggregation results in plain English using existing variables.
import math

def fmt(x):
    try:
        if x is None or (isinstance(x, float) and math.isnan(x)):
            return "None"
        if isinstance(x, float):
            return f"{x:.6g}"
        return str(x)
    except Exception:
        return str(x)

lines = []
lines.append("Traffic KPI aggregation (trace.row_count and log.row_count) summary for full series and incident window 2021-03-25 07:30:00–08:00:00 (UTC+8):")
lines.append("")
# Trace summary
ts = trace_summary.iloc[0]
lines.append("Trace (trace-row_count-sum):")
lines.append(f" - global mean (per-minute sum) = {fmt(ts['mean_full'])}")
lines.append(f" - global std  (per-minute sum) = {fmt(ts['std_full'])}")
lines.append(f" - minutes available in window = {int(ts['window_minutes'])}")
lines.append(f" - earliest significant drop (z <= -2) in window = {fmt(ts['earliest_drop_timestamp(UTC+8)'])}")
lines.append(f" - earliest significant spike (z >= 2) in window = {fmt(ts['earliest_spike_timestamp(UTC+8)'])}")
lines.append(f" - drop_count_in_window = {int(ts['drop_count_in_window'])}, spike_count_in_window = {int(ts['spike_count_in_window'])}")
lines.append("")
# Log summary
ls = log_summary.iloc[0]
lines.append("Log (log-row_count-sum):")
lines.append(f" - global mean (per-minute sum) = {fmt(ls['mean_full'])}")
lines.append(f" - global std  (per-minute sum) = {fmt(ls['std_full'])}")
lines.append(f" - minutes available in window = {int(ls['window_minutes'])}")
lines.append(f" - earliest significant drop (z <= -2) in window = {fmt(ls['earliest_drop_timestamp(UTC+8)'])}")
lines.append(f" - earliest significant spike (z >= 2) in window = {fmt(ls['earliest_spike_timestamp(UTC+8)'])}")
lines.append(f" - drop_count_in_window = {int(ls['drop_count_in_window'])}, spike_count_in_window = {int(ls['spike_count_in_window'])}")
lines.append("")
lines.append("Note: The per-minute window tables are empty (no minutes recorded in the incident window), so no z-scores in-window are available.")
summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [timestamp(UTC+8), sum_value, z]
Index: [],                 series    mean_full     std_full  window_minutes earliest_drop_timestamp(UTC+8) earliest_spike_timestamp(UTC+8)  drop_count_in_window  spike_count_in_window
0  trace-row_count-sum  5743.387097  1514.353491               0                           None                            None                     0                      0, Empty DataFrame
Columns: [timestamp(UTC+8), sum_value, z]
Index: [],               series   mean_full    std_full  window_minutes earliest_drop_timestamp(UTC+8) earliest_spike_timestamp(UTC+8)  drop_count_in_window  spike_count_in_window
0  log-row_count-sum  494.032258  128.493124               0                           None                            None                     0                      0)```
```

In [ ]:
# In[11]:

import pandas as pd

# Reuse existing DataFrames: metric, trace, log (assumed already loaded in the kernel)

# 1) metric.csv: unique kpi_name counts (top 100)
kpi_counts_metric = (
    metric.groupby('kpi_name', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(100)
    .reset_index(drop=True)
)

# metric cmdb_id list (unique, top 100)
metric_cmdbs = pd.Series(metric['cmdb_id'].dropna().unique()).head(100).to_frame(name='cmdb_id')

# 2) trace.csv: unique trace_name counts (top 100)
trace_counts = (
    trace.groupby('trace_name', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(100)
    .reset_index(drop=True)
)

# trace cmdb_id list (unique, top 100)
trace_cmdbs = pd.Series(trace['cmdb_id'].dropna().unique()).head(100).to_frame(name='cmdb_id')

# 3) log.csv: unique log_name counts (top 100)
log_counts = (
    log.groupby('log_name', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(100)
    .reset_index(drop=True)
)

# log cmdb_id list (unique, top 100)
log_cmdbs = pd.Series(log['cmdb_id'].dropna().unique()).head(100).to_frame(name='cmdb_id')

# Return compact summaries (DataFrames)
kpi_counts_metric, metric_cmdbs, trace_counts, trace_cmdbs, log_counts, log_cmdbs

```
Out[11]:
```
```python
# Compose a concise plain-English summary of available KPI / trace / log names and cmdb_ids
def fmt_top(df, name_col, n=5):
    rows = []
    for _, r in df.head(n).iterrows():
        rows.append(f"{r[name_col]} (count={int(r['count'])})")
    return "; ".join(rows)

metric_total_kpis = int(metric['kpi_name'].nunique())
trace_total_names = int(trace['trace_name'].nunique())
log_total_names = int(log['log_name'].nunique())

metric_top5 = fmt_top(kpi_counts_metric, 'kpi_name', 5)
trace_top5 = fmt_top(trace_counts, 'trace_name', 5)
log_top5 = fmt_top(log_counts, 'log_name', 5)

metric_cmdb_list = metric_cmdbs['cmdb_id'].head(20).tolist()
trace_cmdb_list = trace_cmdbs['cmdb_id'].head(20).tolist()
log_cmdb_list = log_cmdbs['cmdb_id'].head(20).tolist()

summary_lines = [
    "Available KPI / trace / log names summary (compact):",
    "",
    f"metric.csv: total unique kpi_name = {metric_total_kpis}",
    f" - Top 5 kpi_name by count: {metric_top5}",
    f" - Unique cmdb_id (top 20): {metric_cmdb_list}",
    "",
    f"trace.csv: total unique trace_name = {trace_total_names}",
    f" - Top 5 trace_name by count: {trace_top5}",
    f" - Unique cmdb_id (top 20): {trace_cmdb_list}",
    "",
    f"log.csv: total unique log_name = {log_total_names}",
    f" - Top log_name by count: {fmt_top(log_counts, 'log_name', 10) if not log_counts.empty else 'none'}",
    f" - Unique cmdb_id (top 20): {log_cmdb_list}",
    "",
    "Use these names to target specific KPIs (e.g., look for 'row_count' in trace/log or specific metric KPI strings)."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  count
0        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    354
1      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    344
2                  OSLinux-OSLinux_ZABBIX_Host_Uptime    344
3       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    343
4          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    343
..                                                ...    ...
95  OSLinux-OSLinux_FILESYSTEM_----boot_FSInodeUse...     94
96  OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSCa...     94
97      OSLinux-OSLinux_FILESYSTEM_----tmp_FSCapacity     94
98  OSLinux-OSLinux_FILESYSTEM_----home_FSInodeUse...     94
99   OSLinux-OSLinux_FILESYSTEM_----tomcat_FSCapacity     94

[100 rows x 2 columns],      cmdb_id
0       IG01
1       IG02
2       MG01
3       MG02
4    Mysql01
..       ...
21  Tomcat03
22  Tomcat04
23  apache01
24  apache02
25  dockerA2

[26 rows x 1 columns],                           trace_name  count
0            trace.self.duration_p95    371
1               trace.self.row_count    371
2           trace.self.duration_mean    371
3         trace.to_MG02.duration_p95    246
4            trace.to_MG02.row_count    246
..                               ...    ...
67  trace.from_dockerA1.duration_p95     60
68   trace.to_Tomcat01.duration_mean     60
69         trace.from_root.row_count      1
70     trace.from_root.duration_mean      1
71      trace.from_root.duration_p95      1

[72 rows x 2 columns],      cmdb_id
0       IG01
1       IG02
2       MG01
3       MG02
4   Tomcat01
5   Tomcat02
6   Tomcat03
7   Tomcat04
8   dockerA1
9   dockerA2
10  dockerB1
11  dockerB2,            log_name  count
0   log.error_count    185
1     log.row_count    185
2  log.gc_pause_max     13,     cmdb_id
0  Tomcat01
1  Tomcat02
2  Tomcat03
3  Tomcat04
4  apache01
5  apache02)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Candidate components
candidates = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04',
              'apache01','apache02','IG01','IG02','MG01','MG02',
              'Mysql01','Mysql02','Redis01','Redis02']

# Ensure timestamp_dt exists and is tz-aware for metric, trace, log (reuse variables from kernel)
for df in (metric, trace, log):
    if 'timestamp_dt' not in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)
    else:
        if df['timestamp_dt'].dt.tz is None:
            df['timestamp_dt'] = df['timestamp_dt'].dt.tz_localize(tz)

# Define incident window (inclusive) in UTC+8
start = pd.Timestamp('2021-03-25 07:30:00').tz_localize(tz)
end   = pd.Timestamp('2021-03-25 08:00:00').tz_localize(tz)

# Helper to filter and prepare display table
def prepare_sample(df, name_col):
    sel = df[
        (df['cmdb_id'].isin(candidates)) &
        (df['timestamp_dt'] >= start) &
        (df['timestamp_dt'] <= end)
    ].copy()
    # compact display columns and sort
    disp = sel.sort_values('timestamp_dt').head(20).loc[:, ['timestamp_dt','cmdb_id', name_col, 'value']].copy()
    disp = disp.rename(columns={'timestamp_dt':'timestamp(UTC+8)'})
    # convert timestamp to compact string for display
    disp['timestamp(UTC+8)'] = disp['timestamp(UTC+8)'].astype(str)
    return disp, int(sel.shape[0])

# Metric sample
metric_window_sel, metric_count_sel = prepare_sample(metric, 'kpi_name')

# Trace sample
trace_window_sel, trace_count_sel = prepare_sample(trace, 'trace_name')

# Log sample
log_window_sel, log_count_sel = prepare_sample(log, 'log_name')

# Return compact results (DataFrames limited to 20 rows and counts)
metric_window_sel, metric_count_sel, trace_window_sel, trace_count_sel, log_window_sel, log_count_sel

```
Out[12]:
```
```python
# Plain-English summary of the sampled rows result (compact)
summary = (
    "Sampled rows for selected components in incident window 2021-03-25 07:30:00–08:00:00 (UTC+8):\n"
    " - metric.csv: 0 rows for the listed components\n"
    " - trace.csv: 0 rows for the listed components\n"
    " - log.csv: 0 rows for the listed components\n\n"
    "No sample rows are available for those components in the specified window. "
    "Possible reasons: those components had no telemetry in that exact window, "
    "or the data is recorded under different cmdb_id values or timestamps. "
    "You can expand the time window or request available cmdb_ids/timestamps in the window for further investigation."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [timestamp(UTC+8), cmdb_id, kpi_name, value]
Index: [], 0, Empty DataFrame
Columns: [timestamp(UTC+8), cmdb_id, trace_name, value]
Index: [], 0, Empty DataFrame
Columns: [timestamp(UTC+8), cmdb_id, log_name, value]
Index: [], 0)```
```

In [ ]:
# In[13]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp_dt exists and is timezone-aware for all files (reuse kernel variables)
for df in (metric, trace, log, error_logs):
    if 'timestamp_dt' not in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)
    else:
        if df['timestamp_dt'].dt.tz is None:
            df['timestamp_dt'] = df['timestamp_dt'].dt.tz_localize(tz)

# Define incident window (inclusive) in UTC+8
start = pd.Timestamp('2021-03-25 07:30:00').tz_localize(tz)
end   = pd.Timestamp('2021-03-25 08:00:00').tz_localize(tz)

# Helper to compute per-file summaries
def per_file_counts(df):
    df_window = df[(df['timestamp_dt'] >= start) & (df['timestamp_dt'] <= end)].copy()
    total_rows = int(df_window.shape[0])
    if total_rows == 0:
        summary_df = pd.DataFrame(columns=['cmdb_id','row_count_in_window'])
    else:
        summary_df = (
            df_window.groupby('cmdb_id', dropna=False)
            .size()
            .reset_index(name='row_count_in_window')
            .sort_values('row_count_in_window', ascending=False)
            .head(50)
            .reset_index(drop=True)
        )
    return summary_df, total_rows

# Compute for each file
metric_counts_window, metric_total_rows_window = per_file_counts(metric)
trace_counts_window, trace_total_rows_window   = per_file_counts(trace)
log_counts_window, log_total_rows_window       = per_file_counts(log)
error_counts_window, error_total_rows_window   = per_file_counts(error_logs)

# Return compact results (DataFrames limited to top 50 and scalar totals)
metric_counts_window, metric_total_rows_window, \
trace_counts_window, trace_total_rows_window, \
log_counts_window, log_total_rows_window, \
error_counts_window, error_total_rows_window

```
Out[13]:
```
```python
# Plain-English summary as a single string using existing kernel variables.
summary = (
    "Per-file row counts in window 2021-03-25 07:30:00–08:00:00 (UTC+8):\n"
    f" - metric.csv: {metric_total_rows_window} rows\n"
    f" - trace.csv:  {trace_total_rows_window} rows\n"
    f" - log.csv:    {log_total_rows_window} rows\n"
    f" - error_logs.csv: {error_total_rows_window} rows\n\n"
    "No cmdb_id-level counts are available because each file has zero rows in the specified window.\n"
    "Conclusion: There is no telemetry data in metric, trace, log, or error_logs for that 30-minute incident window. "
    "Please expand the time range or verify data ingestion if you expect records."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, row_count_in_window]
Index: [], 0, Empty DataFrame
Columns: [cmdb_id, row_count_in_window]
Index: [], 0, Empty DataFrame
Columns: [cmdb_id, row_count_in_window]
Index: [], 0, Empty DataFrame
Columns: [cmdb_id, row_count_in_window]
Index: [], 0)```
```

In [ ]:
# In[14]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp_dt exists and is timezone-aware for all files (reuse existing variables)
for df in (metric, trace, log, error_logs):
    if 'timestamp_dt' not in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)
    else:
        if df['timestamp_dt'].dt.tz is None:
            df['timestamp_dt'] = df['timestamp_dt'].dt.tz_localize(tz)

# Helper to get min/max timestamp strings or None if empty
def min_max_ts(df):
    if df.shape[0] == 0:
        return None, None
    mn = df['timestamp_dt'].min()
    mx = df['timestamp_dt'].max()
    return str(mn), str(mx)

rows = []
for name, df in [('metric.csv', metric), ('trace.csv', trace), ('log.csv', log), ('error_logs.csv', error_logs)]:
    total = int(df.shape[0])
    mn, mx = min_max_ts(df)
    rows.append({
        'filename': name,
        'total_rows': total,
        'min_timestamp(UTC+8)': mn,
        'max_timestamp(UTC+8)': mx
    })

summary_df = pd.DataFrame(rows)

# For metric.csv: top 20 distinct timestamp(UTC+8) values sorted ascending
metric_ts = (
    metric['timestamp_dt']
    .dropna()
    .drop_duplicates()
    .sort_values()
    .head(20)
    .astype(str)
    .reset_index(drop=True)
    .to_frame(name='timestamp(UTC+8)')
)

# Return compact results
summary_df, metric_ts

```
Out[14]:
```
# Plain-English summary using existing results (summary_df and metric_ts) from the kernel.
rows = []
for _, r in summary_df.iterrows():
    rows.append(f"{r['filename']}: total_rows={int(r['total_rows'])}, min={r['min_timestamp(UTC+8)']}, max={r['max_timestamp(UTC+8)']}")

metric_ts_list = metric_ts['timestamp(UTC+8)'].tolist()

summary_lines = [
    "Dataset coverage summary:",
    *rows,
    "",
    "Observation:",
    " - All telemetry in these files is limited to the late-night window around 2021-03-24 23:30 – 2021-03-25 00:00 (UTC+8).",
    " - The incident window you asked about (2021-03-25 07:30:00–08:00:00 UTC+8) has no telemetry in metric, trace, log, or error_logs.",
    "",
    "Metric minute-level sample (first 20 distinct timestamps present, UTC+8):",
    *metric_ts_list,
    "",
    "Conclusion:",
    " - There is no data for the requested incident window in any file. To analyze that window, please confirm data ingestion for that time or provide a different time range that matches available timestamps above."
]

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         filename  total_rows       min_timestamp(UTC+8)       max_timestamp(UTC+8)
0      metric.csv       29735  2021-03-24 23:30:00+08:00  2021-03-25 00:00:00+08:00
1       trace.csv        7014  2021-03-24 23:30:00+08:00  2021-03-25 00:00:00+08:00
2         log.csv         383  2021-03-24 23:30:00+08:00  2021-03-25 00:00:00+08:00
3  error_logs.csv          41  2021-03-24 23:40:08+08:00  2021-03-24 23:44:20+08:00,              timestamp(UTC+8)
0   2021-03-24 23:30:00+08:00
1   2021-03-24 23:31:00+08:00
2   2021-03-24 23:32:00+08:00
3   2021-03-24 23:33:00+08:00
4   2021-03-24 23:34:00+08:00
5   2021-03-24 23:35:00+08:00
6   2021-03-24 23:36:00+08:00
7   2021-03-24 23:37:00+08:00
8   2021-03-24 23:38:00+08:00
9   2021-03-24 23:39:00+08:00
10  2021-03-24 23:40:00+08:00
11  2021-03-24 23:41:00+08:00
12  2021-03-24 23:42:00+08:00
13  2021-03-24 23:43:00+08:00
14  2021-03-24 23:44:00+08:00
15  2021-03-24 23:45:00+08:00
16  2021-03-24 23:46:00+08:00
17  2021-03-24 23:47:00+08:00
18  2021-03-24 23:48:00+08:00
19  2021-03-24 23:49:00+08:00)```
```

In [ ]:
# In[15]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp_dt exists and is timezone-aware for all files (reuse existing variables)
for df in (metric, trace, log, error_logs):
    if 'timestamp_dt' not in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)
    else:
        if df['timestamp_dt'].dt.tz is None:
            df['timestamp_dt'] = df['timestamp_dt'].dt.tz_localize(tz)

# Helper to produce per-date counts and 2021-03-25 checks
def per_date_summary(df, name):
    if df.shape[0] == 0:
        summary = pd.DataFrame(columns=['date(UTC+8)','row_count']).astype({'row_count':'int'})
        has_flag = False
        count_20210325 = 0
    else:
        df = df.copy()
        df['date(UTC+8)'] = df['timestamp_dt'].dt.tz_convert(tz).dt.strftime('%Y-%m-%d')
        grouped = df.groupby('date(UTC+8)', dropna=False).size().reset_index(name='row_count')
        grouped = grouped.sort_values('date(UTC+8)', ascending=False).head(31).reset_index(drop=True)
        summary = grouped
        # check for 2021-03-25
        if '2021-03-25' in grouped['date(UTC+8)'].values:
            has_flag = True
            count_20210325 = int(grouped.loc[grouped['date(UTC+8)']=='2021-03-25', 'row_count'].iloc[0])
        else:
            # If not in the top31 slice, still check entire df grouping to get accurate count for 2021-03-25
            all_grouped = df.groupby('date(UTC+8)', dropna=False).size().reset_index(name='row_count')
            mask = all_grouped['date(UTC+8)']=='2021-03-25'
            if mask.any():
                has_flag = True
                count_20210325 = int(all_grouped.loc[mask,'row_count'].iloc[0])
            else:
                has_flag = False
                count_20210325 = 0
    return summary, has_flag, count_20210325

metric_by_date, metric_has_20210325, metric_count_20210325 = per_date_summary(metric, 'metric.csv')
trace_by_date, trace_has_20210325, trace_count_20210325 = per_date_summary(trace, 'trace.csv')
log_by_date, log_has_20210325, log_count_20210325 = per_date_summary(log, 'log.csv')
error_by_date, error_has_20210325, error_count_20210325 = per_date_summary(error_logs, 'error_logs.csv')

# Return compact outputs
metric_by_date, metric_has_20210325, metric_count_20210325, \
trace_by_date, trace_has_20210325, trace_count_20210325, \
log_by_date, log_has_20210325, log_count_20210325, \
error_by_date, error_has_20210325, error_count_20210325

```
Out[15]:
```
Summary (grouped by date in UTC+8, most recent dates shown):

- metric.csv:
  - 2021-03-25: 1,044 rows
  - 2021-03-24: 28,691 rows
  - Any rows on 2021-03-25? Yes — 1,044 rows.

- trace.csv:
  - 2021-03-25: 177 rows
  - 2021-03-24: 6,837 rows
  - Any rows on 2021-03-25? Yes — 177 rows.

- log.csv:
  - 2021-03-25: 10 rows
  - 2021-03-24: 373 rows
  - Any rows on 2021-03-25? Yes — 10 rows.

- error_logs.csv:
  - 2021-03-24: 41 rows
  - Any rows on 2021-03-25? No — 0 rows.

Notes:
- Each file's table was limited to the most recent dates (up to 31); the lines above show the most recent date(s) present.
- metric, trace, and log have data on 2021-03-25; error_logs does not.

The original code execution output of IPython Kernel is also provided below for reference:

(  date(UTC+8)  row_count
0  2021-03-25       1044
1  2021-03-24      28691, True, 1044,   date(UTC+8)  row_count
0  2021-03-25        177
1  2021-03-24       6837, True, 177,   date(UTC+8)  row_count
0  2021-03-25         10
1  2021-03-24        373, True, 10,   date(UTC+8)  row_count
0  2021-03-24         41, False, 0)```
```